In [ ]:
# 查看GPU配置
!nvidia-smi

# 1. 准备好预训练模型和数据

In [ ]:
# 切换工作目录
%cd /root/autodl-tmp/VITS-fine-tuning
!pwd

In [ ]:
# !python -m pip install phonemizer==3.2.1
# build monotonic align
%cd monotonic_align/
!mkdir monotonic_align
!python setup.py build_ext --inplace
%cd ..

# 创建文件夹
!mkdir ./OUTPUT_MODEL
!mkdir ./custom_character_voice
!mkdir ./pretrained_models
# 拷贝数据与模型
!cp -r ../ESDv2/* ./custom_character_voice
!cp -r ../VITS-fast-fine-tuning/pretrained_models/* ./pretrained_models/
!ls ./custom_character_voice
!ls ./pretrained_models

In [ ]:
# 将所有视频（无论是上传的还是下载的，且必须是.mp4格式）抽取音频
%run scripts/video2audio.py
# 将所有音频（无论是上传的还是从视频抽取的，必须是.wav格式）去噪
!python scripts/denoise_audio.py
# 分割并标注长音频
!python scripts/long_audio_transcribe.py --languages "{PRETRAINED_MODEL}" --whisper_size large-v2
# 标注短音频
!python scripts/short_audio_transcribe.py --languages "{PRETRAINED_MODEL}" --whisper_size large-v2
# 底模采样率可能与辅助数据不同，需要重采样
!python scripts/resample.py

# 2. 划分好训练/测试集的最终标注

In [ ]:
#@markdown ##STEP 3.5
#@markdown 运行该单元格会生成划分好训练/测试集的最终标注，以及配置文件

#@markdown Running this block will generate final annotations for training & validation, as well as config file.

#@markdown 选择是否加入辅助训练数据：/ Choose whether to add auxiliary data:
ADD_AUXILIARY = False #@param {type:"boolean"}
PRETRAINED_MODEL = "CJE" #@param {type:"String"}
#@markdown 辅助训练数据是从预训练的大数据集抽样得到的，作用在于防止模型在标注不准确的数据上形成错误映射。

#@markdown 以下情况请勾选：

#@markdown 总样本少于100条/样本质量一般或较差/样本来自爬取的视频

#@markdown 以下情况可以不勾选：

#@markdown 总样本量很大/样本质量很高/希望加速训练

if ADD_AUXILIARY:
  %run preprocess_v2.py --add_auxiliary_data True --languages "{PRETRAINED_MODEL}"
else:
  %run preprocess_v2.py --languages "{PRETRAINED_MODEL}"

# 3. 开始训练

In [ ]:
# 开始微调模型
Maximum_epochs = "200" #@param {type:"string"}

# 继续之前的模型训练/Continue training from previous checkpoint
CONTINUE = False #@param {type:"boolean"}
if CONTINUE:
  !python finetune_speaker_v2.py -m "./OUTPUT_MODEL" --max_epochs "{Maximum_epochs}" --drop_speaker_embed False --cont True
else:
  !python finetune_speaker_v2.py -m "./OUTPUT_MODEL" --max_epochs "{Maximum_epochs}" --drop_speaker_embed True

# *.Tensorboard
#### <font color="purple">以下命令推荐在终端内执行</font>

In [ ]:
!ps -ef | grep tensorboard | awk '{print $2}' | xargs kill -9 
!tensorboard --port 6007 --logdir "./OUTPUT_MODEL"